#robustez deadband vc top 5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


base = pd.read_excel(
    r"C:\Users\julio\PBL FGV\FGV QUANT\treinee\TraineeFGVquant\bases de dados\economatica-3.xlsx",
    header=3
)

display(base.head())

In [ ]:
precos = base.copy()

precos.columns = ["Data"] + [
    str(col).strip().split("\n")[-1]
    for col in precos.columns[1:]
]

precos["Data"] = pd.to_datetime(precos["Data"], errors="coerce")
precos = precos.dropna(subset=["Data"])
precos = precos.set_index("Data").sort_index()

precos = precos.replace("-", np.nan)
precos = precos.apply(pd.to_numeric, errors="coerce")

# Remover dias em que TODAS as ações estão sem cotação
precos = precos.dropna(how="all")
precos = precos.ffill(limit=1)
precos = precos.loc[:"2023-12-31"]

display(precos.head())

In [ ]:
retornos = precos.pct_change(fill_method=None)
display(retornos.head())

In [ ]:
momentum = precos.shift(21) / precos.shift(252) -1
display(momentum.tail())

ranking = momentum.iloc[-1].dropna().sort_values(ascending=False)
display(ranking.head(30))

In [ ]:
momentum_mensal= momentum.groupby(momentum.index.to_period("M")).tail(1)
momentum_mensal = momentum_mensal.loc["2017":"2023"]
display(momentum_mensal.head())

In [ ]:
top5 = {}
for data, linha in momentum_mensal.iterrows():


 ranking = linha.dropna()

 if len(ranking) < 5:
  continue
 melhores = ranking.nlargest(5).index.tolist()
 top5[data] = melhores
top5 = pd.DataFrame.from_dict(top5, orient="index", columns=["Ativo 1", "Ativo 2", "Ativo 3", "Ativo 4", "Ativo 5"])
display(top5.head())

In [ ]:
pesos_top5 = pd.DataFrame(
    0.0,
    index=top5.index,
    columns=retornos.columns
)

for data, linha in top5.iterrows():

    ativos = linha.tolist()

    pesos_top5.loc[data, ativos] = 0.20

display(pesos_top5.head())


In [ ]:
ordens = pesos_top5.reindex(retornos.index).shift(1)
# Começamos sem posições
pesos_atuais = pd.Series(0.0, index=retornos.columns)

# Armazenar os retornos da carteira
retorno_top5 = pd.Series(index=retornos.index, dtype=float)

for data in retornos.index:

    r = retornos.loc[data]

    # Verificar se faltam retornos de ações compradas
    if r[pesos_atuais > 0].isna().any():
        raise ValueError(f"Retorno ausente em {data}")

    r = r.fillna(0)

    # Calcular o retorno diário da carteira
    retorno_dia = (pesos_atuais * r).sum()

    retorno_top5.loc[data] = retorno_dia

    # Atualizar os pesos após os retornos do dia
    pesos_atuais = pesos_atuais * (1 + r) / (1 + retorno_dia)

    # Executar novas ordens no fechamento, quando houver
    if ordens.loc[data].notna().all():
        pesos_atuais = ordens.loc[data].copy()

# Manter apenas o período de análise
retorno_top5 = retorno_top5.loc["2017":"2023"]

display(retorno_top5.tail())

In [ ]:
patrimonio_top5 = (1 + retorno_top5).cumprod()

patrimonio_top5.plot(figsize=(12, 5))

plt.title("Momentum Top 5 — 2017 a 2023")
plt.ylabel("Patrimônio acumulado (base 1)")
plt.xlabel("Data")
plt.grid(True)
plt.show()

In [ ]:
def criar_deadband(momentum_mensal, limite):
 carteira = []
 historico = {}
 for data, linha in momentum_mensal.iterrows():
  ranking = linha.dropna().sort_values(ascending=False)
  if len(ranking) < 5:
   continue
  permitidos = ranking.head(limite).index
  carteira = [ativo for ativo in carteira if ativo in permitidos]
  for ativo in ranking.index:
   if len(carteira) == 5:
    break
   if ativo not in carteira:
    carteira.append(ativo)
    historico[data] = carteira.copy()
 return pd.DataFrame.from_dict(
        historico,
        orient="index",
        columns=["Ativo 1", "Ativo 2", "Ativo 3", "Ativo 4", "Ativo 5"])

In [ ]:
deadband7 = criar_deadband(momentum_mensal, limite=7)

deadband10 = criar_deadband(momentum_mensal, limite=10)

display(deadband7.head())
display(deadband10.head())

In [ ]:
def criar_pesos(carteira):
    pesos = pd.DataFrame( 0.0,
        index=carteira.index,
        columns=retornos.columns)
    for data, linha in carteira.iterrows():

        ativos = linha.dropna().tolist()

        pesos.loc[data, ativos] = 1 / len(ativos)

    return pesos


In [ ]:
pesos_deadband7 = criar_pesos(deadband7)

pesos_deadband10 = criar_pesos(deadband10)

display(pesos_deadband7.head())
display(pesos_deadband10.head())

In [ ]:
def backtest(pesos_alvo, retornos):

    ordens = pesos_alvo.reindex(retornos.index).shift(1)
    pesos_atuais = pd.Series(0.0, index=retornos.columns)
    historico = []

    for data, r in retornos.iterrows():

        # Verificar retornos ausentes dos ativos comprados
        ativos_faltantes = r[pesos_atuais > 0]
        ativos_faltantes = ativos_faltantes[ativos_faltantes.isna()]

        if not ativos_faltantes.empty:
            print("Data:", data)
            print("Ativos sem retorno:", ativos_faltantes.index.tolist())
            raise ValueError("Retorno ausente")

        r = r.fillna(0)

        # Retorno diário da carteira
        retorno_dia = (pesos_atuais * r).sum()

        # Atualizar pesos após variação dos preços
        pesos_atuais = pesos_atuais * (1 + r) / (1 + retorno_dia)

        # Verificar se há rebalanceamento
        turnover = 0

        if ordens.loc[data].notna().all():

            novos_pesos = ordens.loc[data]

            turnover = (novos_pesos - pesos_atuais).abs().sum()

            pesos_atuais = novos_pesos.copy()

        historico.append({
            "Data": data,
            "Retorno": retorno_dia,
            "Turnover": turnover
        })

    resultado = pd.DataFrame(historico).set_index("Data")

    return resultado.loc["2017":"2023"]

In [ ]:
resultado_top5 = backtest(pesos_top5, retornos)

resultado_db7 = backtest(pesos_deadband7, retornos)

resultado_db10 = backtest(pesos_deadband10, retornos)

comparacao = pd.DataFrame({
    "Top 5": resultado_top5["Retorno"],
    "Deadband 7": resultado_db7["Retorno"],
    "Deadband 10": resultado_db10["Retorno"]
})

display(comparacao.tail())

In [ ]:
patrimonio = (1 + comparacao).cumprod()

patrimonio.plot(figsize=(12, 6))

plt.title("Top 5 vs Deadband — 2017 a 2023")
plt.ylabel("Patrimônio acumulado")
plt.xlabel("Data")
plt.grid(True)
plt.show()

In [ ]:
def calcular_sharpe(retornos):
    media = retornos.mean()
    volatilidade = retornos.std()

    sharpe = (media / volatilidade) * np.sqrt(252)

    return sharpe

In [ ]:
sharpe_top5 = calcular_sharpe(resultado_top5["Retorno"])
sharpe_db7 = calcular_sharpe(resultado_db7["Retorno"])
sharpe_db10 = calcular_sharpe(resultado_db10["Retorno"])

comparacao_sharpe = pd.DataFrame({
    "Estratégia": ["Top 5", "Deadband 7", "Deadband 10"],
    "Sharpe": [sharpe_top5, sharpe_db7, sharpe_db10]
})

display(comparacao_sharpe.round(3))

In [ ]:
def calcular_metricas(resultado):

    r = resultado["Retorno"]

    patrimonio = (1 + r).cumprod()

    retorno_anual = patrimonio.iloc[-1] ** (252 / len(r)) - 1

    drawdown = patrimonio / patrimonio.cummax() - 1

    max_drawdown = drawdown.min()

    turnover_anual = resultado["Turnover"].sum() * 252 / len(r)

    return {
        "Retorno anual": retorno_anual,
        "Max Drawdown": max_drawdown,
        "Turnover anual": turnover_anual
    }


In [ ]:
metricas = pd.DataFrame({
    "Top 5": calcular_metricas(resultado_top5),
    "Deadband 7": calcular_metricas(resultado_db7),
    "Deadband 10": calcular_metricas(resultado_db10)
}).T

display(metricas.round(3))

In [ ]:
def aplicar_custos(resultado, custo):

    retornos_brutos = resultado["Retorno"]

    turnover = resultado["Turnover"]

    retornos_liquidos = (
        (1 + retornos_brutos) * (1 - custo * turnover) - 1
    )

    return retornos_liquidos



In [ ]:
estrategias = {
    "Top 5": resultado_top5,
    "Deadband 7": resultado_db7,
    "Deadband 10": resultado_db10
}

custos = [0, 0.001, 0.003]

resultados_custos = {}

for nome, resultado in estrategias.items():

    sharpes = []

    for custo in custos:

        r_liquido = aplicar_custos(resultado, custo)

        sharpe = calcular_sharpe(r_liquido)

        sharpes.append(sharpe)

    resultados_custos[nome] = sharpes

tabela_custos = pd.DataFrame(
    resultados_custos,
    index=["0%", "0,10%", "0,30%"]
).T

display(tabela_custos.round(3))

In [41]:
periodos = {
    "2017-2020": ("2017", "2020"),
    "2021-2023": ("2021", "2023")
}

resultados_periodos = []

for periodo, (inicio, fim) in periodos.items():

    for nome, resultado in estrategias.items():

        # Selecionar apenas o período
        sub = resultado.loc[inicio:fim]

        # Aplicar custo de 0,10%
        r_liquido = aplicar_custos(sub, 0.001)

        # Calcular Sharpe
        sharpe = calcular_sharpe(r_liquido)

        resultados_periodos.append({
            "Período": periodo,
            "Estratégia": nome,
            "Sharpe": sharpe
        })

comparacao_periodos = pd.DataFrame(resultados_periodos)

display(comparacao_periodos)

,Período,Estratégia,Sharpe
0,2017-2020,Top 5,1.282793
1,2017-2020,Deadband 7,1.171414
2,2017-2020,Deadband 10,1.060955
3,2021-2023,Top 5,0.721777
4,2021-2023,Deadband 7,0.581830
5,2021-2023,Deadband 10,0.675998
